In [ ]:
# ==== 单元格8：权重范数剪枝 ====
import os
import warnings
import torch
import timm
from copy import deepcopy
import time

# 抑制所有警告
warnings.filterwarnings("ignore")

# 设置环境变量
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['GOTO_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ['OPENBLAS_VERBOSE'] = '0'

print("加载原始 UNI 模型...")
model = timm.create_model(
    "vit_large_patch16_224",
    img_size=224,
    init_values=1e-5,
    num_classes=0,
    pretrained=False
)
ckpt = torch.load("./pytorch_model.bin", map_location="cpu")
model.load_state_dict(ckpt, strict=False)
model.eval()

print(f"✅ 原始模型blocks数量: {len(model.blocks)}")

# ====== 快速权重范数剪枝 ======
keep_ratio = 0.75  # 保留 75% Block

start_time = time.time()
model_pruned = deepcopy(model)
model_pruned.eval()

print("🚀 使用快速权重范数剪枝...")

# 计算每个block的权重范数作为重要性指标
def compute_weight_norm_importance(model):
    block_scores = []
    
    for i, block in enumerate(model.blocks):
        # 计算block中所有参数的L2范数
        total_norm = 0
        param_count = 0
        
        for name, param in block.named_parameters():
            if param.requires_grad:
                total_norm += param.norm().item()
                param_count += 1
        
        # 平均范数作为重要性分数
        if param_count > 0:
            importance = total_norm / param_count
        else:
            importance = 0
            
        block_scores.append((i, importance))
        
        if i % 5 == 0:  # 每5个block打印一次进度
            print(f"  计算Block {i}重要性: {importance:.4f}")
    
    return block_scores

# 计算重要性
print("计算block重要性...")
importance_scores = compute_weight_norm_importance(model_pruned)

# 按重要性排序并选择要保留的block
total_blocks = len(model_pruned.blocks)
keep_num = max(1, int(total_blocks * keep_ratio))

# 按重要性排序
importance_scores.sort(key=lambda x: x[1], reverse=True)
important_blocks = [idx for idx, _ in importance_scores[:keep_num]]
important_blocks.sort()  # 保持原始顺序

print(f"保留最重要的 {len(important_blocks)} 个blocks: {important_blocks}")

# 组装新的blocks
new_blocks = []
for idx in important_blocks:
    new_blocks.append(model_pruned.blocks[idx])

model_pruned.blocks = torch.nn.Sequential(*new_blocks)

print(f"✅ 剪枝后模型blocks数量: {len(model_pruned.blocks)}")
print(f"⏱️  剪枝耗时: {time.time() - start_time:.2f}秒")

# 检查 forward 是否正常
example_inputs = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    y = model_pruned(example_inputs)
print("✅ Forward 测试通过, 输出形状:", y.shape)

# 参数量统计
def count_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6

before_params = count_params(model)
after_params = count_params(model_pruned)
print(f"剪枝前参数量: {before_params:.1f}M → 剪枝后: {after_params:.1f}M (压缩率: {(1 - after_params/before_params)*100:.1f}%)")

# 保存剪枝后的模型
torch.save(model_pruned.state_dict(), "./vit_fast_pruned.pth")
print("✅ 快速剪枝模型已保存")

# 导出 ONNX
onnx_path = "./uni_fast_pruned.onnx"
torch.onnx.export(
    model_pruned,
    example_inputs,
    onnx_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}}
)
print(f"✅ 快速剪枝模型 ONNX 导出完成: {onnx_path}")

# 保存重要性分析结果
importance_file = "./block_weight_importance.txt"
with open(importance_file, "w") as f:
    f.write("Block Weight Norm Importance Analysis\n")
    f.write("=" * 50 + "\n")
    for idx, score in importance_scores:
        retained = "✅" if idx in important_blocks else "❌"
        f.write(f"{retained} Block {idx}: Weight Norm Score = {score:.4f}\n")
print(f"✅ Block权重重要性分析已保存: {importance_file}")

/home/ma-user/anaconda3/envs/PyTorch-2.1.0/lib/python3.9/site-packages/torch_npu/utils/collect_env.py:59: UserWarning: Warning: The /home/ma-user/work owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")


/home/ma-user/anaconda3/envs/PyTorch-2.1.0/lib/python3.9/site-packages/torch_npu/__init__.py:234: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default.                      Do not set it to 1 to prevent some unknown errors
  warnings.warn("On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. \


加载原始 UNI 模型...
✅ 原始模型blocks数量: 24
🚀 使用快速权重范数剪枝...
计算block重要性...
  计算Block 0重要性: 20.7819
  计算Block 5重要性: 16.8510
  计算Block 10重要性: 20.0735
  计算Block 15重要性: 26.0257
  计算Block 20重要性: 37.6015
保留最重要的 18 个blocks: [0, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
✅ 剪枝后模型blocks数量: 18
⏱️  剪枝耗时: 1.36秒
✅ Forward 测试通过, 输出形状: torch.Size([1, 1024])
剪枝前参数量: 303.4M → 剪枝后: 227.8M (压缩率: 24.9%)
✅ 快速剪枝模型已保存
✅ 快速剪枝模型 ONNX 导出完成: ./uni_fast_pruned.onnx
✅ Block权重重要性分析已保存: ./block_weight_importance.txt


In [ ]:
# 单元格 8a：稳定版 DKD 蒸馏（fold3，训练集随机抽取0.6）
import os
import time
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import timm
from copy import deepcopy
from tqdm import tqdm
import torch_npu

# ---------------- Config ----------------
device = torch.device("npu:0")
print("使用设备:", device)

root_dir = "./wsi_patches_4fold_final (1)"
batch_size = 16
num_workers = 4
EPOCHS = 30

# ---- DKD 超参 ----
T = 2.0
ALPHA = 0.3
BETA = 0.5
LR = 3e-5
WEIGHT_DECAY = 0.01

# ---------------- 数据路径 ----------------
train_dir = os.path.join(root_dir, "fold3", "train")
val_dir   = os.path.join(root_dir, "fold3", "val")
print("📌 使用的训练数据：", train_dir)
print("📌 使用的验证数据：", val_dir)

# ---------------- transforms ----------------
transform_train = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2,0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

transform_val = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# ---------------- 原始训练集 ----------------
full_train_set = datasets.ImageFolder(root=train_dir, transform=transform_train)
val_set = datasets.ImageFolder(root=val_dir, transform=transform_val)

# ---------------- 随机抽样 0.6 ----------------
SAMPLE_RATIO = 0.6
total_size = len(full_train_set)
sample_size = int(total_size * SAMPLE_RATIO)
sampled_indices = random.sample(range(total_size), sample_size)
train_set = Subset(full_train_set, sampled_indices)

# 保留原始 classes
class_names = full_train_set.classes
num_classes = len(class_names)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader   = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print(f"📌 原始训练集大小: {len(full_train_set)}")
print(f"📌 随机抽取 60% 后训练集大小: {len(train_set)}")
print(f"📌 验证集大小: {len(val_set)}")

# ---------------- 教师模型 ----------------
teacher = timm.create_model(
    "vit_large_patch16_224",
    img_size=224,
    init_values=1e-5,
    num_classes=num_classes,
    pretrained=False
)
ckpt = torch.load("./pytorch_model.bin", map_location=device)
teacher.load_state_dict(ckpt, strict=False)
teacher.to(device)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

# ---------------- 学生模型（剪枝后） ----------------
assert 'model_pruned' in globals(), "❌ 请先运行剪枝单元格，生成 model_pruned"
student = deepcopy(model_pruned)
student.head = nn.Linear(student.num_features, num_classes)
student.to(device)

print("🎯 教师参数量:", sum(p.numel() for p in teacher.parameters()))
print("🎯 学生参数量:", sum(p.numel() for p in student.parameters()))

# ============ DKD 损失函数 ==================
def dkd_loss(student_logits, teacher_logits, labels, alpha=ALPHA, beta=BETA, T=T):
    s_log_T = F.log_softmax(student_logits / T, dim=1)
    t_prob_T = F.softmax(teacher_logits / T, dim=1)

    # 完整 KD KL
    full_kd = F.kl_div(s_log_T, t_prob_T, reduction="batchmean") * (T*T)

    # Target class KD
    gt = labels.unsqueeze(1)
    t_gt = t_prob_T.gather(1, gt).squeeze(1)
    s_log_gt = s_log_T.gather(1, gt).squeeze(1)
    tckd = (t_gt * (torch.log(torch.clamp(t_gt,1e-9)) - s_log_gt)).mean() * (T*T)

    # Non-target KD
    nckd = full_kd - tckd

    # CE + soft CE
    ce = F.cross_entropy(student_logits, labels)
    soft_ce = F.cross_entropy(student_logits / T, teacher_logits.argmax(1))

    return ce + 0.5 * soft_ce + alpha * tckd + beta * nckd

# =================== 训练函数 ======================
optimizer = optim.AdamW(student.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def evaluate(model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            total += y.size(0)
            correct += (pred == y).sum().item()
    return correct / total

# =================== 训练循环 ======================
print("\n🚀 开始 DKD 蒸馏训练（随机抽取 60%）...")
best_acc = 0

for epoch in range(EPOCHS):
    student.train()
    running_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", ncols=100)

    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        with torch.no_grad():
            tea_logits = teacher(x)

        stu_logits = student(x)
        loss = dkd_loss(stu_logits, tea_logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    acc = evaluate(student)
    print(f"Epoch {epoch+1} | Loss={running_loss/len(train_loader):.4f} | Val Acc={acc*100:.2f}%")

    # 保存最佳模型
    if acc > best_acc:
        best_acc = acc
        torch.save(student.state_dict(), "./student_dkd_best.pth")
        print(f"🔥 保存最佳模型，当前验证集精度 {acc*100:.2f}%")

# 保存最终模型
torch.save(student.state_dict(), "./student_dkd_final.pth")
print(f"\n🎉 DKD 蒸馏完成！最佳准确率：{best_acc*100:.2f}%")
student_dkd = student
print("✅ 模型已传到 student_dkd 变量")

In [ ]:
# ==== 修复的单元格9：量化 ====
import torch
import torch_npu
import timm
import numpy as np
import time
import os
import statistics
import sys
from torch.cuda.amp import autocast

# 进度条函数
def progress_bar(current, total, bar_length=50):
    percent = float(current) * 100 / total
    arrow = '-' * int(percent/100 * bar_length - 1) + '>'
    spaces = ' ' * (bar_length - len(arrow))
    
    sys.stdout.write(f'\r进度: [{arrow}{spaces}] {current}/{total} ({percent:.1f}%)')
    sys.stdout.flush()

# 环境
os.environ['PYTORCH_NPU_ALLOC_CONF'] = 'max_split_size_mb:512'
torch.manual_seed(42)
np.random.seed(42) 
if torch_npu.npu.is_available():
    torch_npu.npu.empty_cache()
device = torch.device("npu:0")
print(f"设备: {device}")

# ========== 智能模型加载 ==========
if 'student' in locals():
    print("✅ 检测到内存中的蒸馏学生模型，直接使用 student")
    model = student

elif 'model_pruned' in locals():
    print("⚠️ 没有 student，回退到 model_pruned（剪枝模型）")
    model = model_pruned

else:
    raise ValueError("❌ student 和 model_pruned 都不存在，请先运行剪枝和蒸馏单元格。")

# 验证模型结构
print(f"📊 模型blocks数量: {len(model.blocks)}")

# 将模型移动到设备
model = model.to(device, dtype=torch.float16).eval()

# JIT优化
torch_npu.npu.set_compile_mode(jit_compile=True)
example_input = torch.randn(1, 3, 224, 224).to(device, dtype=torch.float16)
model = torch.jit.trace(model, example_input)
print("✅ PyTorch模型JIT优化完成")

# DataPrefetcher
class DataPrefetcher:
    def __init__(self, loader, device):
        self.loader = iter(loader)
        self.device = device
        self.stream = torch_npu.npu.Stream()
        self.data_load_time = 0  # 新增
        self.preload()
    
    def preload(self):
        load_start = time.time()  # 新增
        try:
            self.next_input, self.next_target = next(self.loader)
        except StopIteration:
            self.next_input = self.next_target = None
            return
        finally:
            self.data_load_time += time.time() - load_start  # 新增
            
        with torch_npu.npu.stream(self.stream):
            self.next_input = self.next_input.to(self.device, non_blocking=True, dtype=torch.float16)
            self.next_target = self.next_target.to(self.device, non_blocking=True)
    
    def next(self):
        if self.device.type == "npu":
            torch_npu.npu.current_stream().wait_stream(self.stream)
        input, target = self.next_input, self.next_target
        self.preload()
        return input, target

# 5次运行
num_runs = 5
run_times, fps_list, inference_scores = [], [], []
total_samples = len(dataset)
wsi_count = len(wsi_labels)
total_batches = len(dataloader)

print(f"开始推理，总批次: {total_batches}, 总样本: {total_samples}")

for run in range(num_runs):
    print(f"\n🔥 Run {run+1}/{num_runs}")
    torch_npu.npu.empty_cache()
    
    start_time = time.time()
    data_load_time = inference_time = post_process_time = 0
    features_list, labels_list = [], []
    peak_memory = 0
    
    prefetcher = DataPrefetcher(dataloader, device)
    imgs, lbls = prefetcher.next()
    batch_count = 0
    
    # 显示进度条
    progress_bar(0, total_batches)
    
    while imgs is not None:
        # 数据加载计时
        load_start = time.time()
        data_load_time += time.time() - load_start
        
        # 推理计时
        infer_start = time.time()
        with torch.no_grad(), autocast(dtype=torch.float16):
            feats = model(imgs)
        inference_time += time.time() - infer_start
        
        # 后处理
        post_start = time.time()
        features_list.append(feats.cpu())
        labels_list.append(lbls.cpu())
        post_process_time += time.time() - post_start
        
        # 内存监控
        current_mem = torch_npu.npu.mem_get_info()[1] / (1024**3)
        peak_memory = max(peak_memory, current_mem)
        
        batch_count += 1
        if batch_count % 50 == 0:
            torch_npu.npu.synchronize()
        
        # 更新进度条（每批次更新）
        progress_bar(batch_count, total_batches)
        
        imgs, lbls = prefetcher.next()
    
    # 完成进度条
    sys.stdout.write('\n')
    sys.stdout.flush()
    
    # 合并
    concat_start = time.time()
    features = torch.cat(features_list, dim=0).numpy()
    labels = torch.cat(labels_list, dim=0).numpy()
    post_process_time += time.time() - concat_start
    
    total_time = time.time() - start_time
    fps = total_samples / total_time
    wsi_per_second = wsi_count / total_time
    inference_score = wsi_per_second * 1000 * 0.5
    
    run_times.append(total_time)
    fps_list.append(fps)
    inference_scores.append(inference_score)
    
    # 保存最后一次
    if run == num_runs - 1:
        save_dir = "distill_after_prunned_2"
        os.makedirs(save_dir, exist_ok=True)
        np.save(f"{save_dir}/bach_features_distill.npy", features)
        np.save(f"{save_dir}/bach_labels_distill.npy", labels)
        print(f"✅ 剪枝模型特征已保存到: {save_dir}")
    
    print(f"⏱️  Time={total_time:.1f}s | 🚀 FPS={fps:.0f} | ⭐ Score={inference_score:.1f}")
    print(f"   📥 Load:{data_load_time:.1f}s | ⚡ Infer:{inference_time:.1f}s | 💾 Post:{post_process_time:.1f}s | 🧠 Mem:{peak_memory:.1f}GB")

# 统计
mean_time = statistics.mean(run_times)
mean_fps = statistics.mean(fps_list)
mean_score = statistics.mean(inference_scores)

print(f"\n🎉 === FP16优化统计 (5 runs) ===")
print(f"平均时间: {mean_time:.1f}s")
print(f"平均FPS: {mean_fps:.0f}")
print(f"平均推理得分: {mean_score:.1f}")
print(f"特征已保存: {save_dir}")

设备: npu:0
✅ 检测到内存中的蒸馏学生模型，直接使用 student
📊 模型blocks数量: 18
...✅ PyTorch模型JIT优化完成
开始推理，总批次: 547, 总样本: 140000

🔥 Run 1/5
进度: [------------------------------------------------->] 547/547 (100.0%)
⏱️  Time=315.1s | 🚀 FPS=444 | ⭐ Score=6.3
   📥 Load:0.0s | ⚡ Infer:21.8s | 💾 Post:136.3s | 🧠 Mem:29.5GB

🔥 Run 2/5
进度: [------------------------------------------------->] 547/547 (100.0%)
⏱️  Time=282.5s | 🚀 FPS=496 | ⭐ Score=7.1
   📥 Load:0.0s | ⚡ Infer:8.8s | 💾 Post:136.4s | 🧠 Mem:29.5GB

🔥 Run 3/5
进度: [------------------------------------------------->] 547/547 (100.0%)
⏱️  Time=283.3s | 🚀 FPS=494 | ⭐ Score=7.1
   📥 Load:0.0s | ⚡ Infer:8.7s | 💾 Post:136.9s | 🧠 Mem:29.5GB

🔥 Run 4/5
进度: [------------------------------------------------->] 547/547 (100.0%)
⏱️  Time=291.2s | 🚀 FPS=481 | ⭐ Score=6.9
   📥 Load:0.0s | ⚡ Infer:8.9s | 💾 Post:136.5s | 🧠 Mem:29.5GB

🔥 Run 5/5
进度: [------------------------------------------------->] 547/547 (100.0%)
✅ 剪枝模型特征已保存到: distill_after_prunned
⏱️  Time=294.5s | 🚀 F